# Day05 下午个人项目：电商用户多维分析

**姓名：** 刘子奥
**专题方向：** A

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "刘子奥"
TOPIC = "A"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 刘子奥
专题： A
输入数据： C:\Users\Lenovo\PyCharmMiscProject\ecommerce-user-analysis-24012399\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： C:\Users\Lenovo\PyCharmMiscProject\ecommerce-user-analysis-24012399\output\day05_analysis


In [2]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 刘子奥
专题： A


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [3]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-12个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,0-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,0-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,0-12个月,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,0-12个月,1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,str
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,str
Gender,str
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [4]:
# TODO 1：定义需要验收的核心字段
core_cols = [
    "CustomerID",
    "Churn",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
]

# TODO 2：完成数据验收表
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
row_data = {
    "总行数": df.shape[0],
    "总列数": df.shape[1],
    "CustomerID重复条数": df["CustomerID"].duplicated().sum(),
    "核心字段总缺失数": df[core_cols].isna().sum().sum(),
    "Churn唯一取值": sorted(df["Churn"].unique())
}
validation = pd.DataFrame([row_data])

# TODO 3：展示验收结果
display(validation)

,总行数,总列数,CustomerID重复条数,核心字段总缺失数,Churn唯一取值
0,5630,22,0,0,"[0, 1]"


In [5]:
# 检查点1：数据结构与核心质量
assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder"
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：每行代表一位电商平台独立用户的完整画像与行为样本，单用户对应一条记录。

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：CustomerID是用户唯一标识编码，数字无业务数值意义，求平均无业务价值，仅作分类标识使用。

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [6]:
# TODO：计算公共基础指标
# 用户数、流失人数、流失率、平均订单数、订单数中位数、
# 平均优惠券数、平均返现、平均App时长、平均满意度、平均距上次下单天数

total_users = df.shape[0]
churn_users = df["Churn"].sum()
overall_churn_rate = churn_users / total_users

avg_order = df["OrderCount"].mean()
median_order = df["OrderCount"].median()
avg_coupon = df["CouponUsed"].mean()
avg_cashback = df["CashbackAmount"].mean()
avg_app_hour = df["HourSpendOnApp"].mean()
avg_satisfaction = df["SatisfactionScore"].mean()
avg_last_order_day = df["DaySinceLastOrder"].mean()

overall_metrics = pd.DataFrame({
    "指标名称": [
        "用户数",
        "流失人数",
        "流失率",
        "平均订单数",
        "订单数中位数",
        "平均优惠券使用数",
        "平均返现金额",
        "平均App使用时长",
        "平均满意度",
        "平均距上次下单天数"
    ],
    "指标数值": [
        total_users,
        churn_users,
        overall_churn_rate,
        avg_order,
        median_order,
        avg_coupon,
        avg_cashback,
        avg_app_hour,
        avg_satisfaction,
        avg_last_order_day
    ]
})

# TODO：展示结果
display(overall_metrics)

,指标名称,指标数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用数,1.72
6,平均返现金额,177.22
7,平均App使用时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [7]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate = 0.16838365896980462
assert overall_churn_rate is not None, \
    "overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：当前样本共有5630名用户，总体流失率为16.8%。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [8]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])


# TODO 1：选择主分组字段
segment_field = "TenureGroup"

# TODO 2：使用groupby + agg完成命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数 = ("CustomerID", "count"),
    平均订单数 = ("OrderCount", "mean"),
    平均App使用时长 = ("HourSpendOnApp", "mean"),
    流失率 = ("Churn", "mean"),
    平均返现金额 = ("CashbackAmount", "mean")
).reset_index()

# 按用户生命周期从小到大排序
segment_analysis = segment_analysis.sort_values(segment_field)

# TODO 3：重置索引、按业务意义排序并展示
display(segment_analysis)

当前专题： A
可选主分组字段： {'TenureGroup'}


,TenureGroup,用户数,平均订单数,平均App使用时长,流失率,平均返现金额
0,0-12个月,3552,2.56,2.94,0.24,159.99
1,12-24个月,1574,3.64,2.94,0.06,200.72
2,24-36个月,500,3.70,2.89,0.00,225.29
3,48-60个月,2,2.50,3.50,0.00,161.50
4,60个月以上,2,1.50,3.50,0.00,291.25


In [9]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：不同用户生命周期分层的用户流失率存在多大差异？

**数据现象：**

> TODO：0-12 个月新用户群体共 3552 人，流失率约 24%；24 个月以上长期用户共 504 人，流失率为0；随用户生命周期变长，分层流失率持续下降。

**可能解释：**

> TODO：用户生命周期时长与留存表现存在负相关，长期使用平台的用户粘性更高，流失意愿更低；新用户阶段流失风险偏高，可能与初次使用体验、活动激励不足相关，该猜想需结合订单、投诉等行为指标进一步验证。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [10]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}

# TODO：选择两个维度
dim_1 = "TenureGroup"
dim_2 = "Complain"

# TODO：双维度分组聚合
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数 = ("CustomerID", "count"),
    流失人数 = ("Churn", "sum"),
    流失率 = ("Churn", "mean"),
    平均订单数 = ("OrderCount", "mean")
).reset_index()

# TODO：新增样本提示列
def sample_label(row):
    if row["用户数"] < 30:
        return "小样本"
    else:
        return "可观察"
cross_analysis["样本提示"] = cross_analysis.apply(sample_label, axis=1)

# TODO：按流失率降序排序展示
cross_analysis = cross_analysis.sort_values("流失率", ascending=False)

# 展示结果
display(cross_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,样本提示
1,0-12个月,1,1019,452,0.44,2.62,可观察
0,0-12个月,0,2533,394,0.16,2.53,可观察
3,12-24个月,1,439,56,0.13,3.27,可观察
2,12-24个月,0,1135,46,0.04,3.79,可观察
4,24-36个月,0,356,0,0.00,3.82,可观察
5,24-36个月,1,144,0,0.00,3.40,可观察
6,48-60个月,0,2,0,0.00,2.50,小样本
7,60个月以上,1,2,0,0.00,1.50,小样本


In [11]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**

> TODO：0-12 个月新用户 + 有投诉 (Complain=1)

**该组合的用户数、流失率和比较对象：**

> TODO：0-12个月有投诉用户共1019人，流失率44%；同生命周期无投诉用户共2533人，流失率仅16%；相同生命周期下，产生投诉的新用户流失风险远高于无投诉新用户。

**是否存在小样本风险：**

> TODO：无小样本风险；判断依据：两组人群用户数量分别为1019、2533，样本量均过千，样本充足，流失率结果具备参考价值。

**为什么不能直接写成因果结论：**

> TODO：当前为静态截面数据，仅能观察投诉行为与流失率存在相关性；未控制订单量、登录频次等其他变量，无法确认投诉是流失的直接诱因，存在其他混淆变量干扰，因此不能判定因果。


## 任务5：输出统计报表（必做）

In [12]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [13]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(5, 6)
通过：cross_analysis.csv，形状为(8, 7)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1

在____用户中，____指标为____，与____相比____。对应证据表：____。

> TODO：在有投诉用户中，指标为流失率，与无投诉用户相比明显偏高。对应证据表：cross_analysis。

### 结论2

> TODO：用户生命周期越长，流失率越低、消费相关指标表现越好，证据表：segment_analysis。

### 结论3

> TODO：不同登录设备的用户 App 使用时长与流失水平存在明显差异，证据表：cross_analysis。

### 分析限制

至少写明一项当前数据不能支持的分析，或一项可能影响结论的限制。

> TODO：缺少用户每日行为时序数据，无法确定投诉行为和用户流失的先后因果逻辑，仅能观测相关性。

### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> TODO：给新产生投诉的用户发放专属优惠券安抚；需要做 AB 测试，选取同期投诉新用户分实验组发券、对照组不发，后续跟踪 30 天两组的留存率数据验证效果。

## 拓展任务（选做）

In [14]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO（选做）
# 2. 转换为绘图长表
melt_cols = ["Tenure", "HourSpendOnApp", "OrderCount", "CashbackAmount"]
long_plot_df = df.melt(
    id_vars=["CustomerID", "Churn", "TenureGroup"],
    value_vars=melt_cols,
    var_name="指标名称",
    value_name="指标数值"
)
# 导出绘图长表
long_plot_df.to_csv(OUTPUT_DIR / "plot_long_table.csv", index=False, encoding="utf-8-sig")
display(long_plot_df.head(10))

,CustomerID,Churn,TenureGroup,指标名称,指标数值
0,50001,1,0-12个月,Tenure,4.00
1,50002,1,0-12个月,Tenure,9.00
2,50003,1,0-12个月,Tenure,9.00
3,50004,1,0-12个月,Tenure,0.00
4,50005,1,0-12个月,Tenure,0.00
5,50006,1,0-12个月,Tenure,0.00
6,50007,1,0-12个月,Tenure,9.00
7,50008,1,0-12个月,Tenure,9.00
8,50009,1,12-24个月,Tenure,13.00
9,50010,1,0-12个月,Tenure,9.00


## 最终检查：GitHub提交前验收

In [15]:
required_files = [
    ROOT / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

本地提交文件检查通过
请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。


### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？

答：
1. 新用户群体里有投诉用户流失率显著高于无投诉用户，投诉是高流失风险信号；且用户生命周期越长，整体流失水平持续走低，长期老客留存表现更好。
2. 核心字段缺失值检查点。
3. “产生投诉会直接导致用户流失”。仅能观测投诉与高流失存在相关性，缺少时序行为数据，无法区分是投诉在先、流失倾向在先，存在混淆变量干扰，不能认定因果。
4. 投诉处理结果字段（是否妥善解决、处理时长）。可区分投诉是否被妥善处理对流失的差异化影响，精准定位是投诉本身还是服务处置差推高流失。
5. 选用cross_analysis投诉 × 生命周期双维交叉统计表。一张分组柱状图可同时展示「有无投诉、不同使用时长」两层流失差异，直观呈现新客投诉高流失核心业务洞察，适合 Flask 网页做核心概览展示。